# Seed `cdp.gtm_events` with simulated GTM events

Generates configurable volumes of realistic Google Tag Manager events and appends them to:

```
nikks_fevm_workspace_7405607030687545.cdp.gtm_events
```

The `eventData` JSON in each row matches the **GA4 server-side container** payload shape that the Zerobus template actually receives (after the GA4 client tag has expanded the simple dataLayer push from `src/cdp_demo_web_shop/ui/lib/gtm.ts`): full GA4 envelope with `all_cookies`, `client_hints`, `client_id`, `event_id`, `event_location`, `ga_session_id`/`ga_session_number`, `ip_override`, `language`, `page_referrer`, `screen_resolution`, plus all `x-ga-*` and `x-sst-system_properties` headers. Ecommerce fields (`items`, `currency`, `value`, `transaction_id`) are flat at the top level — that's how GA4 server-side passes them.

Target table schema (already exists):

```sql
CREATE TABLE nikks_fevm_workspace_7405607030687545.cdp.gtm_events (
  ingestion_time    BIGINT,    -- unix epoch milliseconds
  gtm_container_id  STRING,
  event_name        STRING,
  request_path      STRING,
  request_method    STRING,
  query_string      STRING,
  visitor_region    STRING,
  eventData         STRING     -- JSON blob of the full event payload
);
```

**Event mix:** `pageview` 50% / `view_item` 22% / `add_to_cart` 12% / `purchase` 10% / `sign_up` 4% / `account_deleted` 2%. (`pageview` matches the GA4 legacy name as observed in the live server-side container; the rest follow the GA4 recommended naming.)

**How writes work:** this notebook streams events to the table — one row every `interval_seconds` (default 60) for `run_minutes` (default 60). Optionally, set `backfill_rows > 0` to drop a one-shot historical batch spread over `backfill_days` into the table before the live loop starts. The live loop always appends; `write_mode` only governs the optional backfill batch. Interrupt the live-loop cell at any time to stop early — the row count written so far is printed.

Adjust the widgets below and **Run all**.

In [ ]:
dbutils.widgets.text("interval_seconds", "60", "Seconds between live rows")
dbutils.widgets.text("run_minutes", "60", "Live loop duration (minutes)")
dbutils.widgets.text("backfill_rows", "0", "Backfill batch size (0 = skip)")
dbutils.widgets.text("backfill_days", "30", "Backfill spread (days)")
dbutils.widgets.dropdown("write_mode", "append", ["append", "overwrite"], "Write mode (backfill only)")
dbutils.widgets.text("gtm_container_id", "GTM-K29QPLV2", "GTM container ID")
dbutils.widgets.text("ga_measurement_id", "G-C31T0FRWHZ", "GA4 measurement ID")
dbutils.widgets.text("seed", "42", "Random seed (reproducible runs)")

INTERVAL_SECONDS = float(dbutils.widgets.get("interval_seconds"))
RUN_MINUTES = float(dbutils.widgets.get("run_minutes"))
BACKFILL_ROWS = int(dbutils.widgets.get("backfill_rows"))
BACKFILL_DAYS = int(dbutils.widgets.get("backfill_days"))
WRITE_MODE = dbutils.widgets.get("write_mode")
GTM_CONTAINER_ID = dbutils.widgets.get("gtm_container_id").strip()
GA_MEASUREMENT_ID = dbutils.widgets.get("ga_measurement_id").strip()
GA_MEASUREMENT_ID_SUFFIX = GA_MEASUREMENT_ID.removeprefix("G-")
SEED = int(dbutils.widgets.get("seed"))

TABLE = "serverless_sandbox_avaa82_catalog.cdp.gtm_events"

print(f"interval_seconds  = {INTERVAL_SECONDS}")
print(f"run_minutes       = {RUN_MINUTES}")
print(f"backfill_rows     = {BACKFILL_ROWS:,}")
print(f"backfill_days     = {BACKFILL_DAYS}")
print(f"write_mode        = {WRITE_MODE}  (applies to backfill batch only; live loop always appends)")
print(f"gtm_container_id  = {GTM_CONTAINER_ID}")
print(f"ga_measurement_id = {GA_MEASUREMENT_ID}")
print(f"seed              = {SEED}")
print(f"target table      = {TABLE}")

## Reference data

The product catalogue mirrors `_BOSCH_TOOLS` in `src/cdp_demo_web_shop/backend/seed.py` so simulated `view_item` / `add_to_cart` / `purchase` events reference the same SKUs the web shop actually serves. Product IDs are deterministic UUID5s derived from the product name, so re-running the notebook produces stable IDs.

In [ ]:
import json
import random
import uuid
from datetime import datetime, timedelta, timezone

random.seed(SEED)

# Stable namespace for deterministic product UUIDs
_PRODUCT_NS = uuid.UUID("6ba7b810-9dad-11d1-80b4-00c04fd430c8")


def _product_id(name: str) -> str:
    return str(uuid.uuid5(_PRODUCT_NS, name))


# Mirror of _BOSCH_TOOLS from src/cdp_demo_web_shop/backend/seed.py (active subset)
PRODUCTS = [
    {"id": _product_id(name), "name": name, "price": price}
    for name, price in [
        ("GSR 18V-55", 189.00),
        ("GSB 18V-90 C", 249.00),
        ("GSR 12V-35", 129.00),
        ("PSR 1080 LI", 79.00),
        ("PSB 1800 LI-2", 139.00),
        ("GBH 2-26", 219.00),
        ("GBH 18V-26 F", 379.00),
        ("PBH 2100 RE", 99.00),
        ("GWS 18V-10", 199.00),
        ("PWS 700-115", 59.00),
        ("GWS 22-230 JH", 189.00),
        ("GST 18V-LI S", 169.00),
    ]
]

# Accounts pool: ~100 simulated users with stable IDs
_ACCOUNT_NS = uuid.UUID("8e2c1e94-2a3d-4f6a-9b8d-6f5a3c1e9a2b")
_FIRST_NAMES = [
    "Lukas", "Maximilian", "Felix", "Jonas", "Paul", "Leon", "Elias", "Noah",
    "Finn", "Henry", "Anna", "Marie", "Sophie", "Mia", "Emma", "Lena", "Lara",
    "Sarah", "Laura", "Hannah", "Pieter", "Jan", "Bram", "Sven", "Tomas",
    "Matteo", "Luca", "Giulia", "Sofia", "Chiara", "Carlos", "Diego", "Lucia",
    "Camila", "Pierre", "Louis", "Camille", "Manon", "Oliver", "Harry",
    "Charlotte", "Amelia", "Liam", "Owen", "Mason", "Ethan", "Aria", "Zoe",
]
_SURNAMES = [
    "Schmidt", "Mueller", "Schneider", "Fischer", "Weber", "Meyer", "Wagner",
    "Becker", "Hoffmann", "Schulz", "Koch", "Bauer", "Klein", "Wolf", "Schroeder",
    "Neumann", "Schwarz", "Zimmermann", "Braun", "Hartmann", "Vermeer",
    "Janssen", "DeVries", "Bakker", "Visser", "Rossi", "Bianchi", "Romano",
    "Garcia", "Martinez", "Rodriguez", "Lopez", "Dubois", "Lefevre", "Moreau",
    "Bernard", "Smith", "Jones", "Williams", "Brown", "Taylor", "Wilson",
]
_CITY_COUNTRY = [
    ("Berlin", "DE"), ("Munich", "DE"), ("Hamburg", "DE"), ("Cologne", "DE"),
    ("Frankfurt", "DE"), ("Stuttgart", "DE"), ("Vienna", "AT"), ("Salzburg", "AT"),
    ("Zurich", "CH"), ("Geneva", "CH"), ("Amsterdam", "NL"), ("Rotterdam", "NL"),
    ("Paris", "FR"), ("Lyon", "FR"), ("Rome", "IT"), ("Milan", "IT"),
    ("Madrid", "ES"), ("Barcelona", "ES"), ("London", "GB"), ("Manchester", "GB"),
    ("New York", "US"), ("Chicago", "US"),
]


def _build_accounts(n: int = 100) -> list[dict]:
    rng = random.Random(SEED + 1)
    accounts = []
    for i in range(n):
        first = rng.choice(_FIRST_NAMES)
        last = rng.choice(_SURNAMES)
        city, country = rng.choice(_CITY_COUNTRY)
        acc_id = str(uuid.uuid5(_ACCOUNT_NS, f"{i}-{first}-{last}"))
        accounts.append({
            "id": acc_id,
            "first_name": first,
            "surname": last,
            "email": f"{first.lower()}.{last.lower()}{i}@example.com",
            "city": city,
            "country": country,
        })
    return accounts


ACCOUNTS = _build_accounts(100)

# Page paths the React router can produce (matches src/cdp_demo_web_shop/ui/routes/)
STATIC_PAGES = [
    ("/", "Bosch Shop \u00b7 Catalog", 0.55),
    ("/cart", "Bosch Shop \u00b7 Cart", 0.12),
    ("/accounts", "Bosch Shop \u00b7 Accounts", 0.08),
    ("/purchases", "Bosch Shop \u00b7 Purchases", 0.10),
]
# Remaining 0.15 -> product detail pages, sampled across PRODUCTS

VISITOR_REGIONS = [
    ("DE", 0.40), ("AT", 0.07), ("CH", 0.06), ("NL", 0.07), ("FR", 0.10),
    ("IT", 0.07), ("ES", 0.06), ("GB", 0.08), ("US", 0.06), ("unknown", 0.03),
]

# --- GA4 server-side payload reference data ----------------------------------

# Each USER_AGENTS entry is paired with a Client Hints object so client_hints
# matches the UA string GA4 sees server-side.
USER_AGENTS = [
    {
        "ua": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36",
        "hints": {
            "architecture": "x86", "bitness": "64",
            "full_version_list": [
                {"brand": "Google Chrome", "version": "143.0.7499.170"},
                {"brand": "Chromium", "version": "143.0.7499.170"},
                {"brand": "Not A(Brand", "version": "24.0.0.0"},
            ],
            "mobile": False, "model": "",
            "platform": "Windows", "platform_version": "15.0.0", "wow64": False,
        },
    },
    {
        "ua": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36",
        "hints": {
            "architecture": "arm", "bitness": "64",
            "full_version_list": [
                {"brand": "Google Chrome", "version": "143.0.7499.170"},
                {"brand": "Chromium", "version": "143.0.7499.170"},
                {"brand": "Not A(Brand", "version": "24.0.0.0"},
            ],
            "mobile": False, "model": "",
            "platform": "macOS", "platform_version": "15.7.1", "wow64": False,
        },
    },
    {
        "ua": "Mozilla/5.0 (X11; Linux x86_64; rv:124.0) Gecko/20100101 Firefox/124.0",
        "hints": {
            "architecture": "x86", "bitness": "64",
            "full_version_list": [{"brand": "Firefox", "version": "124.0"}],
            "mobile": False, "model": "",
            "platform": "Linux", "platform_version": "6.5.0", "wow64": False,
        },
    },
    {
        "ua": "Mozilla/5.0 (iPhone; CPU iPhone OS 17_4 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.4 Mobile/15E148 Safari/604.1",
        "hints": {
            "architecture": "arm", "bitness": "64",
            "full_version_list": [{"brand": "Safari", "version": "17.4"}],
            "mobile": True, "model": "iPhone",
            "platform": "iOS", "platform_version": "17.4.0", "wow64": False,
        },
    },
    {
        "ua": "Mozilla/5.0 (Linux; Android 14; Pixel 8) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Mobile Safari/537.36",
        "hints": {
            "architecture": "arm", "bitness": "64",
            "full_version_list": [
                {"brand": "Google Chrome", "version": "124.0.6367.118"},
                {"brand": "Chromium", "version": "124.0.6367.118"},
                {"brand": "Not A(Brand", "version": "24.0.0.0"},
            ],
            "mobile": True, "model": "Pixel 8",
            "platform": "Android", "platform_version": "14.0.0", "wow64": False,
        },
    },
]

# Sub-regions per country code (used for event_location.region).
SUB_REGIONS_BY_COUNTRY = {
    "DE": ["BE", "BY", "BW", "NW", "HE", "HH", "SN", "NI"],
    "AT": ["W", "OE", "ST", "T", "S"],
    "CH": ["ZH", "GE", "BE", "VD"],
    "NL": ["NH", "ZH", "UT", "NB"],
    "FR": ["IDF", "PAC", "ARA", "OCC"],
    "IT": ["LOM", "LAZ", "VEN", "CAM"],
    "ES": ["MD", "CT", "AN", "VC"],
    "GB": ["ENG", "SCT", "WLS", "NIR"],
    "US": ["CA", "NY", "TX", "FL", "IL", "WA"],
    "unknown": ["unknown"],
}

# Locale per country (BCP-47-ish, lowercased to match the captured sample).
LOCALES_BY_COUNTRY = {
    "DE": "de-de", "AT": "de-at", "CH": "de-ch", "NL": "nl-nl",
    "FR": "fr-fr", "IT": "it-it", "ES": "es-es", "GB": "en-gb",
    "US": "en-us", "unknown": "en-us",
}

# Realistic IPv4 prefixes per country (kept short — first two octets per range).
IP_PREFIXES_BY_COUNTRY = {
    "DE": ["78.94", "217.255", "85.214", "188.40"],
    "AT": ["80.110", "194.232"],
    "CH": ["80.218", "194.230"],
    "NL": ["82.169", "94.157"],
    "FR": ["80.236", "92.184"],
    "IT": ["79.42", "93.146"],
    "ES": ["81.36", "88.27"],
    "GB": ["81.106", "137.83"],
    "US": ["104.28", "137.83", "73.158"],
    "unknown": ["198.51", "203.0"],
}

# Weighted pool of common viewport-style screen resolutions (desktop + mobile).
SCREEN_RESOLUTIONS = [
    ("1920x1080", 0.18),
    ("1440x900", 0.14),
    ("1728x1117", 0.12),
    ("2560x1440", 0.08),
    ("1366x768", 0.10),
    ("1536x864", 0.08),
    ("390x844", 0.10),
    ("412x915", 0.07),
    ("375x812", 0.07),
    ("414x896", 0.06),
]

# Plausible-looking GA4 constants captured from the live server-side container.
# These rarely vary across events for a given site; safe to keep as defaults.
GA_GTM_VERSION = "45je6171v9238789775z89237708170za20gzb9237708170zd9237708170"
GA_TAG_EXP = (
    "103116026~103200004~104527906~104528500~104684208~104684211~105391253~"
    "115938465~115938469~116184924~116184926~116514483~116744866"
)
GA_GCD = "13l3l3l3l1l1"

# x-sst-system_properties fields — fixed-ish across requests.
X_SST_DEFAULTS = {
    "gcsub": "region1",
    "navt": "r",
    "sw_exp": "1",
    "ude": "0",
}

EVENT_WEIGHTS = [
    ("pageview", 0.50),
    ("view_item", 0.22),
    ("add_to_cart", 0.12),
    ("purchase", 0.10),
    ("sign_up", 0.04),
    ("account_deleted", 0.02),
]

BASE_URL = "https://shop.bosch.example"

## Generators

One helper per GA4 event name. Each returns the `(request_path, query_string, eventData_dict)` triple that the row builder assembles into the final row. The `eventData` shapes match what `trackPageView` / `trackViewItem` / `trackAddToCart` / `trackPurchase` / `trackSignUp` / `trackAccountDeleted` push to `window.dataLayer` in `lib/gtm.ts`.

In [ ]:
def _weighted_choice(rng: random.Random, weighted: list[tuple]) -> object:
    r = rng.random()
    acc = 0.0
    for value, w in weighted:
        acc += w
        if r <= acc:
            return value
    return weighted[-1][0]


def _diurnal_timestamp(rng: random.Random, days_back: int) -> datetime:
    """Sample a UTC timestamp within the last `days_back` with diurnal weighting (09-22 local-ish heavier)."""
    now = datetime.now(timezone.utc)
    start = now - timedelta(days=days_back)
    span_seconds = int((now - start).total_seconds())

    day_offset_seconds = rng.randint(0, span_seconds)
    candidate = start + timedelta(seconds=day_offset_seconds)
    if rng.random() < 0.75:
        hour = rng.randint(9, 22)
        candidate = candidate.replace(
            hour=hour, minute=rng.randint(0, 59), second=rng.randint(0, 59),
            microsecond=rng.randint(0, 999) * 1000,
        )
    return candidate


def _pick_product(rng: random.Random) -> dict:
    return rng.choice(PRODUCTS)


def _pick_account(rng: random.Random) -> dict:
    return rng.choice(ACCOUNTS)


def _item(product: dict, quantity: int | None = None) -> dict:
    item = {
        "item_id": product["id"],
        "item_name": product["name"],
        "price": product["price"],
        "currency": "EUR",
    }
    if quantity is not None:
        item["quantity"] = quantity
    return item


def _pick_request_path(rng: random.Random) -> tuple[str, str]:
    """Return (request_path, page_title) consistent with the React router."""
    r = rng.random()
    acc = 0.0
    for path, title, w in STATIC_PAGES:
        acc += w
        if r <= acc:
            return path, title
    product = _pick_product(rng)
    return f"/product/{product['id']}", f"Bosch Shop \u00b7 {product['name']}"


def _maybe_user_id(rng: random.Random, anonymous_share: float = 0.20) -> str | None:
    """Typical 80% logged-in / 20% anonymous split."""
    if rng.random() < anonymous_share:
        return None
    return _pick_account(rng)["id"]


# ---- Per-event overlays ----------------------------------------------------
# Each returns (request_path, page_title, overlay_dict) where overlay is the
# event-specific fields that are layered on TOP of the GA4 envelope built in
# the next cell. GA4 server-side carries ecommerce flat at the top level
# (items, currency, value) - NOT nested under "ecommerce: {...}".


def build_pageview(rng: random.Random) -> tuple[str, str, dict]:
    path, title = _pick_request_path(rng)
    user_id = _maybe_user_id(rng)
    overlay: dict = {}
    if user_id:
        overlay["user_id"] = user_id
    return path, title, overlay


def build_view_item(rng: random.Random) -> tuple[str, str, dict]:
    product = _pick_product(rng)
    path = f"/product/{product['id']}"
    title = f"Bosch Shop \u00b7 {product['name']}"
    user_id = _maybe_user_id(rng)
    overlay: dict = {
        "currency": "EUR",
        "value": product["price"],
        "items": [_item(product)],
    }
    if user_id:
        overlay["user_id"] = user_id
    return path, title, overlay


def build_add_to_cart(rng: random.Random) -> tuple[str, str, dict]:
    product = _pick_product(rng)
    quantity = rng.choices([1, 1, 1, 2, 2, 3, 4], k=1)[0]
    # Add-to-cart happens from the catalog (~60%) or product detail (~40%).
    if rng.random() < 0.6:
        path = "/"
        title = "Bosch Shop \u00b7 Catalog"
    else:
        path = f"/product/{product['id']}"
        title = f"Bosch Shop \u00b7 {product['name']}"
    user_id = _maybe_user_id(rng, anonymous_share=0.10)
    overlay: dict = {
        "currency": "EUR",
        "value": round(product["price"] * quantity, 2),
        "items": [_item(product, quantity=quantity)],
    }
    if user_id:
        overlay["user_id"] = user_id
    return path, title, overlay


def build_purchase(rng: random.Random) -> tuple[str, str, dict]:
    account = _pick_account(rng)
    n_lines = rng.choices([1, 1, 2, 2, 3, 4, 5], k=1)[0]
    chosen = rng.sample(PRODUCTS, k=min(n_lines, len(PRODUCTS)))
    items: list[dict] = []
    subtotal = 0.0
    for p in chosen:
        qty = rng.choices([1, 1, 1, 2, 3], k=1)[0]
        items.append(_item(p, quantity=qty))
        subtotal += p["price"] * qty
    shipping = rng.choices([0.00, 4.95, 9.95], weights=[0.4, 0.4, 0.2], k=1)[0]
    tax = round(subtotal * 0.19, 2)  # German VAT
    value = round(subtotal + shipping, 2)
    overlay = {
        "user_id": account["id"],
        "transaction_id": str(uuid.uuid4()),
        "currency": "EUR",
        "value": value,
        "tax": tax,
        "shipping": shipping,
        "items": items,
    }
    return "/cart", "Bosch Shop \u00b7 Cart", overlay


def build_sign_up(rng: random.Random) -> tuple[str, str, dict]:
    account = _pick_account(rng)
    overlay = {
        "method": "form",
        "user_id": account["id"],
        "email": account["email"],
        "first_name": account["first_name"],
        "surname": account["surname"],
        "city": account["city"],
        "country": account["country"],
    }
    return "/accounts", "Bosch Shop \u00b7 Accounts", overlay


def build_account_deleted(rng: random.Random) -> tuple[str, str, dict]:
    account = _pick_account(rng)
    overlay = {
        "method": "self_service",
        "user_id": account["id"],
        "email": account["email"],
        "first_name": account["first_name"],
        "surname": account["surname"],
        "city": account["city"],
        "country": account["country"],
    }
    return "/accounts", "Bosch Shop \u00b7 Accounts", overlay


BUILDERS = {
    "pageview": build_pageview,
    "view_item": build_view_item,
    "add_to_cart": build_add_to_cart,
    "purchase": build_purchase,
    "sign_up": build_sign_up,
    "account_deleted": build_account_deleted,
}

In [ ]:
# ---- GA4 server-side envelope helpers --------------------------------------


def _random_ipv4(rng: random.Random, country: str) -> str:
    prefix = rng.choice(IP_PREFIXES_BY_COUNTRY.get(country, IP_PREFIXES_BY_COUNTRY["unknown"]))
    return f"{prefix}.{rng.randint(0, 255)}.{rng.randint(1, 254)}"


def _build_all_cookies(rng: random.Random, client_id: str, ga_session_id: int,
                       ga_session_number: int, event_unix_s: int,
                       engagement_seconds: int) -> str:
    """Synthesize the `_ga` + `_ga_<MEAS_ID_SUFFIX>` cookie pair as a single header string."""
    h = rng.randint(10**9, 9 * 10**9)
    return (
        f"_ga=GA1.1.{client_id}; "
        f"_ga_{GA_MEASUREMENT_ID_SUFFIX}=GS2.1."
        f"s{ga_session_id}$o{ga_session_number}$g1$t{event_unix_s}"
        f"$j{engagement_seconds}$l0$h{h}"
    )


def build_envelope(
    rng: random.Random,
    event_name: str,
    event_dt: datetime,
    request_path: str,
    page_title: str,
) -> dict:
    """Construct the GA4 server-side envelope for a single event row."""
    event_unix_s = int(event_dt.timestamp())
    event_unix_ms = int(event_dt.timestamp() * 1000)

    # Country / region / IP / locale picked together for consistency.
    country = _weighted_choice(rng, VISITOR_REGIONS)
    region_code = rng.choice(SUB_REGIONS_BY_COUNTRY[country])
    language = LOCALES_BY_COUNTRY[country]
    ip_override = _random_ipv4(rng, country)

    # User-agent + matching client_hints.
    ua_entry = rng.choice(USER_AGENTS)

    # client_id: <rand10>.<unix_seconds_first_seen> where first-seen is up to
    # 30 days before the event itself (returning visitor pattern).
    first_seen_unix = event_unix_s - rng.randint(0, 30 * 86400)
    client_id_random = rng.randint(10**9, 9 * 10**9)
    client_id = f"{client_id_random}.{first_seen_unix}"

    # Session start within the last 30 minutes of this event.
    ga_session_id = event_unix_s - rng.randint(0, 1800)
    ga_session_number = rng.choices([1, 1, 1, 2, 2, 3, 4, 5, 10, 20], k=1)[0]

    engagement_ms = rng.randint(50, 30000)
    engagement_s = max(1, engagement_ms // 1000)

    page_location = f"{BASE_URL}{request_path}"
    # 70% direct (referrer == location), 30% from another internal page.
    if rng.random() < 0.7:
        page_referrer = page_location
    else:
        other_path, _ = _pick_request_path(rng)
        page_referrer = f"{BASE_URL}{other_path}"

    screen_resolution = _weighted_choice(rng, SCREEN_RESOLUTIONS)

    # x-ga-page_id: int milliseconds, roughly aligned to session start.
    x_ga_page_id = ga_session_id * 1000 + rng.randint(0, 999999)
    # x-sst tft mirrors page_id in real captured payloads.
    tft = x_ga_page_id

    etld_cc = country.lower() if country in ("DE", "AT", "CH", "FR", "IT", "ES", "NL") else (
        "co.uk" if country == "GB" else ("com" if country == "US" else "com")
    )
    etld = f"google.{etld_cc}"

    return {
        "all_cookies": _build_all_cookies(rng, client_id, ga_session_id, ga_session_number,
                                          event_unix_s, engagement_s),
        "client_hints": ua_entry["hints"],
        "client_id": client_id,
        "engagement_time_msec": engagement_ms,
        "event_id": f"{event_unix_ms}_{rng.randint(10**14, 9 * 10**14)}",
        "event_location": {"country": country, "region": region_code},
        "event_name": event_name,
        "ga_session_id": str(ga_session_id),
        "ga_session_number": ga_session_number,
        "ip_override": ip_override,
        "language": language,
        "page_location": page_location,
        "page_referrer": page_referrer,
        "page_title": page_title,
        "screen_resolution": screen_resolution,
        "user_agent": ua_entry["ua"],
        "x-ga-are": "1",
        "x-ga-dma": "0",
        "x-ga-ecid": str(rng.randint(10**9, 9 * 10**9)),
        "x-ga-gcd": GA_GCD,
        "x-ga-gtm_version": GA_GTM_VERSION,
        "x-ga-measurement_id": GA_MEASUREMENT_ID,
        "x-ga-mp2-frm": "0",
        "x-ga-mp2-seg": "1",
        "x-ga-mp2-tag_exp": GA_TAG_EXP,
        "x-ga-npa": "0",
        "x-ga-page_id": x_ga_page_id,
        "x-ga-protocol_version": "2",
        "x-ga-pscdl": "noapi",
        "x-ga-request_count": rng.randint(1, 60),
        "x-ga-system_properties": {"eu": [rng.randint(20, 40)], "tu": "BA"},
        "x-ga-tfd": rng.randint(10_000, 500_000),
        "x-sst-system_properties": {
            "etld": etld,
            "gcsub": X_SST_DEFAULTS["gcsub"],
            "lpc": str(rng.randint(10_000_000, 99_999_999)),
            "navt": X_SST_DEFAULTS["navt"],
            "request_start_time_ms": event_unix_ms + rng.randint(50, 1500),
            "sw_exp": X_SST_DEFAULTS["sw_exp"],
            "tft": tft,
            "ude": X_SST_DEFAULTS["ude"],
        },
    }


def build_row(rng: random.Random, event_dt: datetime) -> dict:
    """Generate a single event row at the given timestamp."""
    event_name = _weighted_choice(rng, EVENT_WEIGHTS)
    path, title, overlay = BUILDERS[event_name](rng)

    envelope = build_envelope(rng, event_name, event_dt, path, title)
    # Overlay event-specific fields. event_name / page_location / etc. already
    # come from the envelope; per-event overlays only contribute payload
    # additions (items, value, user_id, method, ...).
    event_data = {**envelope, **overlay}

    return {
        "ingestion_time": int(event_dt.timestamp() * 1000),
        "gtm_container_id": GTM_CONTAINER_ID,
        "event_name": event_name,
        "request_path": "/gtm",
        "request_method": "POST",
        "query_string": "gtm_debug=1" if rng.random() < 0.05 else "",
        "visitor_region": envelope["event_location"]["country"],
        "eventData": json.dumps(event_data, ensure_ascii=False, separators=(",", ":")),
    }


def generate_backfill_rows(n: int, days_back: int) -> list[dict]:
    """Generate `n` row dicts spread over the last `days_back` days."""
    rng = random.Random(SEED)
    return [build_row(rng, _diurnal_timestamp(rng, days_back)) for _ in range(n)]


# Small preview sample (independent RNG) used only by the preview cells below.
_preview_rng = random.Random(SEED + 7)
preview_rows = [build_row(_preview_rng, _diurnal_timestamp(_preview_rng, 1)) for _ in range(12)]
print(f"Preview sample: {len(preview_rows)} rows")

In [ ]:
from pyspark.sql.types import (
    StructType, StructField, LongType, StringType,
)

schema = StructType([
    StructField("ingestion_time", LongType(), nullable=False),
    StructField("gtm_container_id", StringType(), nullable=False),
    StructField("event_name", StringType(), nullable=False),
    StructField("request_path", StringType(), nullable=False),
    StructField("request_method", StringType(), nullable=False),
    StructField("query_string", StringType(), nullable=True),
    StructField("visitor_region", StringType(), nullable=True),
    StructField("eventData", StringType(), nullable=False),
])

preview_df = spark.createDataFrame(preview_rows, schema=schema)
print(f"Preview DataFrame rows: {preview_df.count():,}")
preview_df.printSchema()

## Preview

A few sample rows per event type — useful to eyeball the `eventData` JSON before writing.

In [ ]:
display(preview_df.limit(10))

In [ ]:
# Print one prettified eventData JSON per event type so the payload shape is easy to inspect.
# Generate a slightly larger preview pool here to maximise the chance of covering every event.
_shape_rng = random.Random(SEED + 11)
_shape_rows = [build_row(_shape_rng, _diurnal_timestamp(_shape_rng, 1)) for _ in range(200)]

seen: dict[str, dict] = {}
for r in _shape_rows:
    seen.setdefault(r["event_name"], r)
    if len(seen) == len(EVENT_WEIGHTS):
        break

for name, _w in EVENT_WEIGHTS:
    sample = seen.get(name)
    print(f"--- {name} ---")
    if sample is None:
        print("(no sample generated)")
    else:
        print(json.dumps(json.loads(sample["eventData"]), indent=2, ensure_ascii=False))
    print()

## Write to Unity Catalog

Two phases:

1. **Backfill batch** (only if `backfill_rows > 0`): generates `backfill_rows` events spread over the last `backfill_days` days and writes them in one Spark job. The `write_mode` widget (`append` / `overwrite`) applies here. Skipped entirely when `backfill_rows = 0`.
2. **Live loop**: every `interval_seconds`, generate one event with `event_dt = now()` and append it to the table as a single-row write. Runs until `run_minutes` elapses, or until the cell is interrupted. Each row prints a one-line status (`[N] timestamp event_name region`).

Interrupting the cell (stop button / `Ctrl+C`) stops the loop cleanly and prints the total rows written this run.

In [ ]:
import time

written = 0

if BACKFILL_ROWS > 0:
    backfill = generate_backfill_rows(BACKFILL_ROWS, BACKFILL_DAYS)
    bf_df = spark.createDataFrame(backfill, schema=schema)
    bf_df.write.mode(WRITE_MODE).saveAsTable(TABLE)
    written += len(backfill)
    print(f"Backfill: wrote {len(backfill):,} rows to {TABLE} (mode={WRITE_MODE})")
else:
    print("Backfill skipped (backfill_rows=0)")

live_rng = random.Random(SEED + 1009)
deadline = time.monotonic() + RUN_MINUTES * 60
expected_rows = int(RUN_MINUTES * 60 / INTERVAL_SECONDS) if INTERVAL_SECONDS > 0 else 0
print(f"Live loop: every {INTERVAL_SECONDS}s for {RUN_MINUTES} min (~{expected_rows} rows)")

try:
    while time.monotonic() < deadline:
        now = datetime.now(timezone.utc)
        row = build_row(live_rng, now)
        (spark.createDataFrame([row], schema=schema)
              .write.mode("append").saveAsTable(TABLE))
        written += 1
        print(f"[{written:>5}] {now.isoformat(timespec='seconds')}  "
              f"{row['event_name']:<16} {row['visitor_region']}")
        remaining = deadline - time.monotonic()
        if remaining <= 0:
            break
        time.sleep(min(INTERVAL_SECONDS, remaining))
except KeyboardInterrupt:
    print("Interrupted by user")

print(f"Done. Total rows written this run: {written:,}")

## Verification

Per-event row counts and time-range coverage in the target table — confirms the write landed and the data spans the requested window.

In [ ]:
verify_df = spark.sql(f"""
    SELECT
        event_name,
        COUNT(*)                                      AS rows,
        MIN(ingestion_time)                           AS min_ingestion_time_ms,
        MAX(ingestion_time)                           AS max_ingestion_time_ms,
        from_unixtime(MIN(ingestion_time) / 1000)     AS min_event_time_utc,
        from_unixtime(MAX(ingestion_time) / 1000)     AS max_event_time_utc
    FROM {TABLE}
    GROUP BY event_name
    ORDER BY rows DESC
""")
display(verify_df)